# 组合优化基本用法

给定预期收益、风险和约束，求取满足条件的证券权重。按任务选择入口：

| 场景 | 入口 |
|---|---|
| 已有数组，只求一次权重 | `CVXPC(...).solve()` |
| 输入包含因子、按期生成权重 | `OptimizationModel` + `MakeOptimizedPortfolio` |
| 在计算图中组合目标、约束和诊断 | `PortfolioProblem` + `CVXPortfolio` |
| 换手与费用取决于实际成交持仓 | [组合优化策略](组合优化策略.ipynb) 中的 `OptimizedStrategy` |

下面先用确定性本地小数据求解，再介绍参数与图接口。目标和约束的后端数学协议见 [原理和细节](原理和细节.ipynb)。

## 最小示例：求解数组权重

示例使用固定随机种子构造正定协方差，设置全额投资和非负权重。`solve()` 返回权重和求解信息，应先确认成功再使用权重。

In [ ]:
import numpy as np
import cvxpy as cvx

# DEMO 数据
np.random.seed(0)
nID = 10
Mask = np.full(shape=(nID,), fill_value=True, dtype=bool)
ExpectedReturn = np.random.randn(nID)
Cov = np.cov(np.random.randn(100 * nID, nID), rowvar=False)
P0 = np.random.rand(nID)
P0 = P0 / np.sum(P0)
Bmk = np.random.rand(nID)
Bmk = Bmk / np.sum(Bmk)

In [ ]:
# 最小方差组合
from QuantStudio.PortfolioConstructor.CVXPC import CVXPC
from QuantStudio.PortfolioConstructor.BasePC import MeanVarianceObjective, BudgetConstraint, WeightConstraint

Objective = MeanVarianceObjective(
    mask=Mask, expected_return=ExpectedReturn, p0=P0, cov=Cov,
    args={"ExpectedReturnCoef": 0, "RiskAversionCoef": 1}
)
ConstraintList = [
    WeightConstraint(mask=Mask, up_limit=1, down_limit=0),
    BudgetConstraint(mask=Mask, args={"UpLimit": 1, "DownLimit": 1})
]

PC = CVXPC(mask=Mask, objective=Objective, constraints=ConstraintList,
           args={"OptimOption": {"solver": cvx.SCIP, "verbose": False}})
Portfolio, Info = PC.solve()

print(np.where(np.abs(Portfolio) > 1e-8, Portfolio, 0))
print(Info)

## 可组合模型与计算图

`OptimizationModel` 将目标、约束和后端组合起来。`ObjectiveSpec` 与 `ConstraintSpec` 使用原有组件的参数模型进行校验：`args` 中填写组件参数，`inputs` 中绑定组件构造函数的数据入参。`mask` 由整个模型统一注入，不能在单个组件中另行指定。

```python
model = OptimizationModel(
    objective=ObjectiveSpec(目标类型, args=目标参数, inputs=数据绑定),
    constraints=[ConstraintSpec(约束类型, args=约束参数, inputs=数据绑定), ...],
    backend=CVXPC,
)
weights = MakeOptimizedPortfolio(model)(mask=股票池因子)
```

`MakeOptimizedPortfolio` 自动将模型中的 Factor、RiskTable 提取为显式图依赖，同一节点只绑定一次；模型参数中仅保留输入位置和固定值。每个时点在完整证券截面上组装问题并求解，返回权重因子。`MakeMeanVariancePortfolio` 保留原有便捷调用，与通用算子使用相同的模型构造和求解路径。

| 组件输入 | 可绑定的数据 |
|---|---|
| `expected_return`、`p0`、`bmk`、`specific_risk`、`budget` | 因子、带证券标签的 Series，或当前截面顺序的数组；标量会广播 |
| `up_limit`、`down_limit` | 常量、因子或证券向量 |
| `cov` | 带证券行列标签的 DataFrame、当前截面顺序的二维数组，或 RiskTable |
| `factor_cov`、`factor_data`、`specific_risk` | 可共同绑定同一个 FactorRT，按字段请求并对齐；因子协方差和暴露也可使用固定 DataFrame |
| `factor_data` | 数值暴露还可绑定一个因子或一组因子，按列组成暴露矩阵；类别暴露传入单个分类因子 |

有标签的数据按证券 ID 对齐，协方差必须覆盖完整截面。数组按当前截面顺序解释。证券查询子集通过 `ids` 指定，完整优化截面通过 `section_ids` 指定。`factor_args` 支持 `SectionIDs`、`CalcDTRuler` 和缓存等原有因子参数；`on_failure="nan"` 可在求解失败时记录警告并返回整期缺失，默认抛出异常。

下面的实例同时使用逐期权重上限和总换手约束。`TurnoverConstraint` 的 `AmtMultiple=0` 表示仅使用换手比例限制，无需提供成交额和账户金额。`p0` 是显式输入，算子不会自动将上一期目标权重当作实际持仓。


In [ ]:
# -*- coding: utf-8 -*-
"""组合收益目标、预算、逐期权重上限和换手约束，生成图中的权重因子。"""
import datetime as dt

import numpy as np
import pandas as pd

from QuantStudio.Factor.Factor import DataFactor
from QuantStudio.PortfolioConstructor.api import (
    OptimizationModel, ObjectiveSpec, ConstraintSpec, MakeOptimizedPortfolio,
    MeanVarianceObjective, BudgetConstraint, WeightConstraint, TurnoverConstraint,
)



ids = ["A", "B"]
dts = [dt.datetime(2026, 1, 1), dt.datetime(2026, 1, 2)]
cov = pd.DataFrame(np.diag([1., 4.]), index=ids, columns=ids)
alpha = DataFactor(pd.DataFrame([[.1, .2], [.3, .1]], index=dts, columns=ids))
cap = DataFactor(pd.DataFrame([[.35, 1.], [.8, 1.]], index=dts, columns=ids))
p0 = DataFactor(pd.DataFrame([[.3, .7]] * 2, index=dts, columns=ids))

model = OptimizationModel(
    objective=ObjectiveSpec(MeanVarianceObjective,
        args={"ExpectedReturnCoef": 1., "RiskAversionCoef": 1.},
        inputs={"expected_return": alpha, "cov": cov}),
    constraints=[
        ConstraintSpec(BudgetConstraint, args={"UpLimit": 1., "DownLimit": 1.}),
        ConstraintSpec(WeightConstraint, inputs={"up_limit": cap, "down_limit": 0.}),
        ConstraintSpec(TurnoverConstraint, args={"AmtMultiple": 0., "UpLimit": .2},
                       inputs={"p0": p0}),
    ],
)
weights = MakeOptimizedPortfolio(model)(factor_args={"Name": "目标权重", "SectionIDs": ids})
result = weights.readData(ids=ids, section_ids=ids, dts=dts, dt_ruler=dts)
np.testing.assert_allclose(result.values, [[.35, .65], [.4, .6]], atol=1e-5)
print(result)

## 均值方差目标

`MeanVarianceObjective` 用收益、风险与交易成本惩罚平衡组合效用。`ExpectedReturnCoef=0`、`RiskAversionCoef=1` 表示最小方差；非零收益项需要 `expected_return`，换手惩罚需要 `p0`。

风险输入可以是完整 `cov`，也可以是 `factor_cov + factor_data + specific_risk`。后者的 `specific_risk` 为标准差，构造协方差时取平方。完整公式、参数、相对基准与换手示例见 [均值方差模型](均值方差模型.ipynb)。

### RiskBudgetObjective — 风险预算优化目标

详见 [风险预算模型](./风险预算模型.ipynb)。

**参数列表**（通过 `args` 传入）：无额外参数。

```python
RiskBudgetObjective(
    mask,                # array(shape=(n,)), 股票池
    budget=None,         # array(shape=(n,)), 风险预算，None 表示等风险预算（风险平价）
    factor_cov=None,     # 因子协方差阵
    factor_data=None,    # 因子暴露矩阵
    specific_risk=None,  # 特异性风险
    cov=None,            # 证券协方差阵
    args={},
    config_file=None
)
```

## 选择约束

预算约束控制总权重，权重约束控制个股上下限，暴露、波动率和换手约束控制对应风险或交易范围。默认 `DropPriority=-1` 不允许释放约束；非负值表示求解失败后可以按优先级释放，结果必须同时检查 `ReleasedConstraint`。

约束字典的数学协议见 [原理和细节](原理和细节.ipynb)。

### BudgetConstraint — 预算约束

限制组合的总权重。对应数学形式：$\mathbf{1}^T(\mathbf{w}-\mathbf{w}_b) \in [a_{down}, a_{up}]$

**参数列表**（通过 `args` 传入）：

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `UpLimit` | `float` | `1.0` | 总权重上限 |
| `DownLimit` | `float` | `1.0` | 总权重下限 |
| `Benchmark` | `bool` | `False` | 是否相对基准 |

```python
# 全额投资约束（权重之和 = 1）
BudgetConstraint(mask=Mask, args={"UpLimit": 1, "DownLimit": 1})

# 相对基准的预算约束
BudgetConstraint(mask=Mask, bmk=Bmk, args={"UpLimit": 0.05, "DownLimit": -0.05, "Benchmark": True})
```

### WeightConstraint — 权重约束

限制单个证券的权重范围。对应数学形式：$\mathbf{a} \le \mathbf{w}-\mathbf{w}_b \le \mathbf{b}$

**参数列表**（通过 `args` 传入）：

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `Benchmark` | `bool` | `False` | 是否相对基准 |

```python
# 纯多头约束（权重在 [0, 1] 之间）
WeightConstraint(mask=Mask, up_limit=1, down_limit=0)

# 个股权重上限（可为标量或向量）
WeightConstraint(mask=Mask, up_limit=0.1)

# 相对基准的权重偏离约束
WeightConstraint(mask=Mask, bmk=Bmk, up_limit=0.02, down_limit=-0.02, args={"Benchmark": True})
```

### FactorExposeConstraint — 因子暴露约束

限制组合相对于基准的因子暴露。对应数学形式：$\mathbf{x}^T(\mathbf{w}-\mathbf{w}_b) \in [a_{down}, a_{up}]$

**参数列表**（通过 `args` 传入）：

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `FactorType` | `Literal["数值型", "类别型"]` | `"数值型"` | 因子类型，类别型会自动转换为哑变量 |
| `UpLimit` | `float` | `1.0` | 因子暴露上限 |
| `DownLimit` | `float` | `1.0` | 因子暴露下限 |
| `Benchmark` | `bool` | `False` | 是否相对基准 |

```python
# 风格中性约束（行业因子暴露与基准一致）
FactorExposeConstraint(
    mask=Mask, factor_data=IndustryDummy,
    bmk=Bmk,
    args={"UpLimit": 0, "DownLimit": 0, "Benchmark": True, "FactorType": "类别型"}
)
```

### VolatilityConstraint — 波动率约束

限制组合的波动率（或跟踪误差）。对应数学形式：$(\mathbf{w}-\mathbf{w}_b)^T\Sigma(\mathbf{w}-\mathbf{w}_b) \le \sigma^2$

**参数列表**（通过 `args` 传入）：

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `UpLimit` | `float` | `0.06` | 波动率上限（年化） |
| `Benchmark` | `bool` | `False` | 是否相对基准（为 True 时限制的是跟踪误差） |

```python
# 组合年化波动率不超过 10%
VolatilityConstraint(mask=Mask, cov=Cov, args={"UpLimit": 0.10})

# 跟踪误差不超过 3%
VolatilityConstraint(mask=Mask, bmk=Bmk, cov=Cov, args={"UpLimit": 0.03, "Benchmark": True})
```

### TurnoverConstraint — 换手率约束

限制组合换手率。可以限制总换手、总买入、总卖出、个券买卖等。

**参数列表**（通过 `args` 传入）：

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `ConstraintType` | `Literal["总换手限制", "总买入限制", "总卖出限制", "买卖限制", "买入限制", "卖出限制"]` | `"总换手限制"` | 限制类型 |
| `AmtMultiple` | `float` | `1.0` | 成交额倍数（个券限制时使用） |
| `UpLimit` | `float` | `0.7` | 限制上限 |

```python
# 总换手不超过 50%
TurnoverConstraint(mask=Mask, p0=P0, args={"ConstraintType": "总换手限制", "UpLimit": 0.5})

# 总买入不超过 30%（卖出不受限）
TurnoverConstraint(mask=Mask, p0=P0, args={"ConstraintType": "总买入限制", "UpLimit": 0.3})

# 个券买卖不超过日均成交额的 10%
TurnoverConstraint(mask=Mask, p0=P0, wealth=1e6, amt=DailyAmt, args={"ConstraintType": "买卖限制", "AmtMultiple": 0.1})
```

### ExpectedReturnConstraint — 预期收益约束

限制组合的预期收益下限。对应数学形式：$\boldsymbol{\mu}^T(\mathbf{w}-\mathbf{w}_b) \ge a$

**参数列表**（通过 `args` 传入）：

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `DownLimit` | `float` | `0.0` | 预期收益下限 |
| `Benchmark` | `bool` | `False` | 是否相对基准 |

```python
# 组合预期收益不低于 5%（年化）
ExpectedReturnConstraint(mask=Mask, expected_return=ExpectedReturn, args={"DownLimit": 0.05})
```

### NonZeroNumConstraint — 非零权重数目约束

限制组合中持仓数量。对应数学形式：$\operatorname{nnz}(\mathbf{w}-\mathbf{w}_b) \le N$

> **注意**：该约束引入了 0-1 整数变量，会将问题转换为混合整数规划（MIP），求解难度和时间大幅增加。

**参数列表**（通过 `args` 传入）：

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `UpLimit` | `int` | `150` | 持仓数量上限 |
| `Benchmark` | `bool` | `False` | 是否相对基准 |

```python
# 持仓不超过 50 只
NonZeroNumConstraint(mask=Mask, args={"UpLimit": 50})
```

## CVXPC — 组合构造器

`CVXPC` 是基于 CVXPY 的凸优化组合构造器，继承自 `PortfolioConstructor`。它接收一个优化目标和一组约束条件，通过 `solve()` 方法求解最优组合权重。

### 构造参数

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `OptimOption` | `dict` | `{}` | 优化选项，传递给 `cvxpy.Problem.solve(**OptimOption)` |

### solve() 方法

```python
PC = CVXPC(mask=Mask, objective=Objective, constraints=ConstraintList, args={"OptimOption": {...}})
Portfolio, Info = PC.solve()
```

**返回值**：
- `Portfolio`：`array(shape=(n,))`，完整长度（与原始 mask 长度一致）的最优权重向量，不在股票池中的 ID 对应位置为 NaN
- `Info`：`dict`，求解信息：
  - `status`：1 表示求解成功，0 表示失败
  - `msg`：求解器返回的状态信息
  - `solver_name`：使用的求解器名称
  - `solve_time`：求解耗时（秒）
  - `setup_time`：问题构建耗时（秒）
  - `num_iters`：求解迭代次数
  - `ReleasedConstraint`：被松弛的约束条件列表

### 求解器选择

通过 `OptimOption` 中的 `solver` 参数选择求解器：

| 求解器 | 适用场景 | 特点 |
|--------|----------|------|
| `cvx.SCIP` | MIP 问题（含 NonZeroNum 约束） | 开源，支持整数规划 |
| `cvx.CLARABEL` | 锥规划（RiskBudget 等） | 开源，支持指数锥 |
| `cvx.OSQP` | 二次规划 | 开源，ADMM 算子分裂 |
| `cvx.ECOS` | 锥规划 | 开源，内点法 |

```python
# 使用 OSQP 求解器
PC = CVXPC(mask=Mask, objective=Objective, constraints=ConstraintList, 
           args={"OptimOption": {"solver": cvx.OSQP, "verbose": False}})

# 使用 SCIP 求解器（含整数约束时推荐）
PC = CVXPC(mask=Mask, objective=Objective, constraints=ConstraintList,
           args={"OptimOption": {"solver": cvx.SCIP, "verbose": True}})
```

## 模型对比与选择

| 模型 | 数学形式 | 输入要求 | 特点 | 适用场景 |
|------|----------|----------|------|----------|
| 最小方差 | $\min \mathbf{w}^T\Sigma\mathbf{w}$ | 仅需协方差阵 | 对参数估计敏感度较低，但持仓可能集中 | 纯风险控制 |
| 均值方差 | $\max \gamma\boldsymbol{\mu}^T\mathbf{w} - \frac{\lambda}{2}\mathbf{w}^T\Sigma\mathbf{w}$ | 预期收益 + 协方差阵 | 收益估计误差影响大，权重对输入敏感 | 收益-风险权衡 |
| 风险预算 | $\min \sum(\mathcal{RC}_i - b_i\mathcal{R})^2$ | 协方差阵 + 风险预算 | 风险分散化，权重较均衡 | 风险分散化投资 |

详见各专题文档：
- [均值方差模型](./均值方差模型.ipynb)
- [风险预算模型](./风险预算模型.ipynb)
- [组合优化策略](./组合优化策略.ipynb)

### 后端能力与扩展

默认后端为 CVXPC。均值方差目标支持组合现有预算、权重、暴露、收益、换手和波动率等约束；`DropPriority` 直接放在各约束的 `args` 中。所有数据相同的输入会共享图依赖。

当前 CVXPC 的风险预算和最大分散化算法自带非负、归一化处理，但尚不支持传入通用约束。通过组合接口为它们指定约束会在构建模型时明确报错，不会静默忽略。最大夏普率目标也需要提供支持它的后端。

新增目标可以继承 `OptimizationObjective` 并实现 `genObjective()`；新增约束继承 `Constraint` 并实现 `genConstraint()`。兼容已有数学形式的约束可直接组合；新数学形式需要后端实现。通过 `OptimizationModel(..., backend=自定义PortfolioConstructor子类)` 使用自定义求解后端，其 `_solve()` 及结果约定与原有构造器一致：返回 `(权重数组或None, {"status": 1或0, ...})`。后端可实现类方法 `validate_model(objective_type, constraint_types)` 声明支持范围。

在逐期账户策略中，也可以用当前数组组装模型并调用 `model.solve(mask, ids)`，直接复用相同的组件与后端。模型中含图节点时，应通过通用算子运行，或显式提供 `resolve` 回调取得当期数据。


## 原生 Node 接口（首版）

Node 模式把目标、约束、问题、求解和结果出口建模为具名依赖节点，便于在 Python 中组合，也为计算图编辑器提供同一套参数和端口声明。数组接口与便捷算子继续可用，三种入口共用既有数学转换和 CVXPY 求解实现。

| 入口 | 典型接口 | 适用方式 |
| --- | --- | --- |
| 当期数组 | `MeanVarianceObjective`、`CVXPC.solve()` | 调用方已经取得当前截面数据 |
| 便捷因子算子 | `MakeMeanVariancePortfolio`、`MakeOptimizedPortfolio` | 将已配置模型转为独立逐期的权重因子 |
| 原生 Node | `PortfolioProblem`、`CVXPortfolio`、`PortfolioWeights` | 将目标、约束和数据来源分别配置、连线 |
| 连续账户 | `OptimizedStrategy` | 每次决策绑定实际现金与持仓，再求解和执行 |

```mermaid
flowchart LR
    A[Factor 与 RiskTable] --> B[目标和约束 Node]
    B --> C[PortfolioProblem]
    C --> D[CVXPortfolio]
    D --> E[PortfolioWeights]
    D --> F[PortfolioDiagnostics]
    C --> G[OptimizedStrategy]
    G --> H[账户与实际权重 Factor]
```

### 参数、依赖与生命周期

`args` 保存模型参数，例如风险厌恶系数、预算上下限、调仓日期和求解选项；`.bind()` 绑定具名依赖，例如收益因子、风险表、股票池及约束集合。因子依赖需要 `Factor`，常量可用 `DataFactor` 包装；协方差来源使用 Risk 模块的 `DataRiskTable` 或其他 `RiskTable/FactorRT`。

`.template()` 返回可部分绑定的 `NodeTemplate`，`.bind()` 返回新模板，`.build()` 才产生完整节点。构图不会读取业务数据或求解。引擎在 `init_compute` 传播日期与完整截面，在准备阶段执行来源 I/O，再在计算阶段组装当期问题并求解。

### 最小完整示例

下面的示例直接持有内存风险与收益数据。目标和约束节点分别构造，再汇入问题和求解节点。

In [ ]:
import datetime as dt
import numpy as np
import pandas as pd
from QuantStudio.Factor.Factor import DataFactor
from QuantStudio.Risk.api import DataRiskTable
from QuantStudio.PortfolioConstructor.api import (
    MeanVarianceObjectiveNode, BudgetConstraintNode, WeightConstraintNode,
    PortfolioProblem, CVXPortfolio, PortfolioWeights, PortfolioDiagnostics,
)

node_dates = [dt.datetime(2026, 1, day) for day in (1, 2)]
node_ids = ['A', 'B']
node_risk = DataRiskTable(pd.DataFrame(np.diag([1., 4.]), index=node_ids, columns=node_ids))
node_alpha = DataFactor(pd.DataFrame([[.1, .2], [.3, .1]], index=node_dates, columns=node_ids))
node_objective = MeanVarianceObjectiveNode.template(args={'ExpectedReturnCoef': 1.}).bind(
    expected_return=node_alpha, risk_table=node_risk).build()
node_budget = BudgetConstraintNode.template().build()
node_limits = WeightConstraintNode.template().build()
node_problem_template = PortfolioProblem.template().bind(constraints=(node_budget, node_limits))
assert 'objective' in node_problem_template.missing_dependencies
node_problem = node_problem_template.bind(objective=node_objective).build()
node_solution = CVXPortfolio.template(args={'SectionIDs': node_ids}).bind(problem=node_problem).build()
node_weights = PortfolioWeights.template().bind(source=node_solution).build()
node_values = node_weights.readData(ids=node_ids, dts=node_dates, dt_ruler=node_dates)
np.testing.assert_allclose(node_values, [[.78, .22], [.84, .16]], atol=1e-6)
node_values

### 组件与具名端口

| 节点 | 主要依赖 | 主要参数或行为 |
| --- | --- | --- |
| `MeanVarianceObjectiveNode` | `risk_table`、`expected_return`、`bmk`、`p0` | 收益/风险系数、相对基准、费用惩罚 |
| `RiskBudgetObjectiveNode` | `risk_table`、可选 `budget` 因子 | 风险预算专用算法 |
| `MaxDiversificationObjectiveNode` | `risk_table` | 最大分散化专用算法 |
| `BudgetConstraintNode` | 可选 `bmk` | `UpLimit`、`DownLimit`、`Benchmark` |
| `WeightConstraintNode` | `up_limit`、`down_limit`、可选 `bmk` | 默认上下限为 1 和 0 |
| `FactorExposeConstraintNode` | `factor_data=(因子, ...)`、可选 `bmk` | 数值或分类暴露限制 |
| `VolatilityConstraintNode` | `risk_table`、可选 `bmk` | 波动率限制 |
| `ExpectedReturnConstraintNode` | `expected_return`、可选 `bmk` | 组合收益限制 |
| `TurnoverConstraintNode` | 外部 `p0`、`wealth`、`amt`（按配置需要） | 持仓来源、换手及成交额限制 |
| `NonZeroNumConstraintNode` | 可选 `bmk` | 证券数量限制，依赖整数求解器 |
| `PortfolioProblem` | 一个 `objective`、有序 `constraints`、可选 `mask` | 同一完整截面的当期问题 |

当前 CVX 后端的均值方差目标可使用通用约束；风险预算和最大分散化专用算法不接受额外约束，具体用法见[风险预算模型](风险预算模型.ipynb)。这些目标也不能用于当前连续账户，因为账户必须施加现金与交易硬限制。

### 完整截面和结果子集

`CVXPortfolio.args.SectionIDs` 指定优化的完整证券顺序；未指定时使用运行上下文的截面。目标、约束和风险数据必须能对齐这个截面。`readData(ids=...)` 只选择结果列，不能把组合问题拆成逐证券求解。

In [ ]:
node_subset = node_weights.readData(ids=['B'], section_ids=node_ids,
                                  dts=node_dates, dt_ruler=node_dates)
np.testing.assert_allclose(node_subset['B'], node_values['B'], atol=1e-6)
node_scaled = (node_weights * 2).readData(ids=node_ids, section_ids=node_ids, dts=node_dates)
np.testing.assert_allclose(node_scaled, node_values * 2, atol=1e-6)
node_subset

### 同时读取权重和诊断

`CVXPortfolio` 的运行结果是 `PortfolioResult`，包含权重表和逐期 `SolveResult`。`PortfolioWeights` 是可连接其他因子的 Factor；`PortfolioDiagnostics` 输出诊断字典。将多个出口放在同一次 `Engine.run()` 中，会复用同一来源的求解结果。

独立调用两次 `.readData()` 会分别启动运行，不承诺跨运行复用。下面显式传入 `FactorContext` 与局部日期、ID 请求。

In [ ]:
from QuantStudio.Core.CalcEngine import Engine
from QuantStudio.Factor.Factor import FactorContext, FactorLocalContext

node_diagnostics = PortfolioDiagnostics.template().bind(source=node_solution).build()
node_local = FactorLocalContext(DTs=node_dates, IDs=node_ids)
node_run_weights, node_run_diagnostics = Engine().run(
    [node_weights, node_diagnostics], FactorContext(DTRuler=node_dates, SectionIDs=node_ids),
    fwd_data_list=[node_local, node_local])
np.testing.assert_allclose(node_run_weights, node_values, atol=1e-6)
assert all(result.status == 'success' for result in node_run_diagnostics.values())
pd.DataFrame({date: {'状态': result.status, '最大约束残差': result.info.get('max_residual')}
              for date, result in node_run_diagnostics.items()}).T

### 调仓、失败与状态来源

| 配置 | 当前行为 |
| --- | --- |
| `CVXPortfolio.RebalanceDTs=None` | 对所有请求日期独立求解 |
| `RebalanceDTs=日期序列` | 仅指定日期求解，其余权重 NaN、状态 `skipped` |
| `OnFailure='raise'` | 求解失败终止，默认行为 |
| `OnFailure='nan'` | 记录失败并输出整期 NaN，不覆盖输入无效异常 |
| `OptimOption` | 传递 CVXPY 求解器选项 |
| `Tolerance` | 生效约束允许的最大残差，默认 `1e-5` |
| `AcceptInaccurate` | 默认不接受不精确最优状态 |

`SolveResult.status` 为 `success`、`relaxed`、`skipped` 或 `failed`。`relaxed` 表示按约束的 `DropPriority` 释放了一部分约束，详情保存在 `info['ReleasedConstraint']`；负值不可释放，非负数值小的先释放，同值一起释放。成功结果还会检查生效约束残差。

需要初始权重时必须声明 `HoldingSource='外部因子'` 并连接 `p0`，或声明 `HoldingSource='账户状态'`，由连续账户在决策时绑定。`WealthSource` 对净资产使用同样的来源声明。纯独立求解无法满足账户状态需求；也不能同时声明账户状态并连接同名外部因子。见[均值方差模型](均值方差模型.ipynb)与[组合优化策略](组合优化策略.ipynb)的示例。

Node 模式支持顺序引擎与 ParallelEngine。并行时参照 SectionOperation，依赖读取完整截面、独立求解按日期分工，结果通过共享文件缓存交换；连续账户由一个进程按时序推进。QSWeb 的 portfolio_constructor 组件目录提供目标、约束、问题和结果节点，风险来源页提供风险库、数据风险表和滚动协方差风险表。

运行示例为 `scripts/demo_portfolio_nodes.py`；设计方案现位于 `docs/plans/PortfolioConstructor节点化与连续优化方案.md`。本 Notebook 记录实际 API，设计文档中的后续能力应以实现状态为准。

### 完整截面的并行执行

并行运行需要已启动且各进程可访问的文件缓存，例如 FeatherFactorCache；每次任务使用独立缓存目录，并配置相同的 DTRuler 和 PIDList。上游普通因子按证券分片计算，组合目标读取完整截面；CVXPortfolio 与滚动协方差按日期分工，每个日期使用全部证券。直接输出组合节点或风险表时，引擎向各进程传递同一完整请求，防止矩阵被拆成对角块。

每个进程发布自己的日期分片，无日期任务的进程也参与同步；全部发布后，各进程取得完整结果，权重因子再投影到所需证券。同一运行的权重、诊断等出口复用求解结果。缓存按节点实例、完整截面、RunID 与 DataVersion 隔离，不提供跨运行续算。相关节点或依赖失败时，同步等待会退出，FailMode='skip' 可继续计算独立出口。

下面复用前文的 node_solution 和 node_weights，验证并行与顺序输出一致。Windows 独立脚本应在 if __name__ == '__main__' 入口内启动引擎。

In [ ]:
from tempfile import TemporaryDirectory
from QuantStudio.Core.ParallelEngine import ParallelEngine
from QuantStudio.Factor.Factor import FactorContext, FactorLocalContext
from QuantStudio.Factor.FactorCache import FeatherFactorCache

parallel_pids = ['0', '1']
with TemporaryDirectory(prefix='qs-portfolio-demo-') as parallel_directory:
    parallel_cache = FeatherFactorCache(args={
        'CacheDir': parallel_directory, 'StartMode': 'new',
        'DTRuler': node_dates, 'PIDs': parallel_pids,
    })
    with parallel_cache, FactorContext(DTRuler=node_dates, SectionIDs=node_ids,
                                      PIDList=parallel_pids, DataCache=parallel_cache) as parallel_context:
        with ParallelEngine() as parallel_engine:
            parallel_solution, parallel_weights = parallel_engine.run(
                [node_solution, node_weights], parallel_context,
                fwd_data_list=[FactorLocalContext(DTs=node_dates, IDs=node_ids) for _ in range(2)])
pd.testing.assert_frame_equal(parallel_solution.weights, node_values)
pd.testing.assert_frame_equal(parallel_weights, node_values)
parallel_weights